In [ ]:
# %% Cell 1 — Import & Cấu hình
import random

GOAL_STATE     = (1, 2, 3, 4, 5, 6, 7, 8, 0)
GOAL_POSITIONS = {val: (i // 3, i % 3) for i, val in enumerate(GOAL_STATE)}
MAX_STEPS      = 1000

print("✅ Cell 1 xong!")

In [28]:
# %% Cell 2 — Heuristic Manhattan Distance
def manhattan(state):
    total = 0
    for i, val in enumerate(state):
        if val == 0:
            continue
        r, c   = i // 3, i % 3
        gr, gc = GOAL_POSITIONS[val]
        total += abs(r - gr) + abs(c - gc)
    return total

print("✅ Cell 2 xong!")

✅ Cell 2 xong!


In [29]:
# %% Cell 3 — Sinh trạng thái ngẫu nhiên (luôn giải được)
def random_state(steps=20):
    state     = list(GOAL_STATE)
    zero      = state.index(0)
    prev_zero = -1
    for _ in range(steps):
        r, c       = zero // 3, zero % 3
        candidates = []
        if r > 0: candidates.append(zero - 3)
        if r < 2: candidates.append(zero + 3)
        if c > 0: candidates.append(zero - 1)
        if c < 2: candidates.append(zero + 1)
        candidates = [x for x in candidates if x != prev_zero] or candidates
        nxt        = random.choice(candidates)
        state[zero], state[nxt] = state[nxt], state[zero]
        prev_zero, zero = zero, nxt
    return tuple(state)

print("✅ Cell 3 xong!")

✅ Cell 3 xong!


In [30]:
# %% Cell 4 — Lấy các nước đi hợp lệ & Hiển thị bảng
def get_moves(state):
    moves = []
    s     = list(state)
    zero  = s.index(0)
    r, c  = zero // 3, zero % 3
    dirs  = {
        'UP':    (r > 0, zero - 3),
        'DOWN':  (r < 2, zero + 3),
        'LEFT':  (c > 0, zero - 1),
        'RIGHT': (c < 2, zero + 1),
    }
    for direction, (valid, nxt) in dirs.items():
        if valid:
            ns = s[:]
            ns[zero], ns[nxt] = ns[nxt], ns[zero]
            moves.append((tuple(ns), direction))
    return moves

def print_board(state, title=""):
    if title:
        print(f"\n  {title}")
    print("  ┌───────┐")
    for row in range(3):
        cells = [' ' if state[row*3+col] == 0
                 else str(state[row*3+col])
                 for col in range(3)]
        print(f"  │ {' '.join(cells)} │")
    print("  └───────┘")

print("✅ Cell 4 xong!")

✅ Cell 4 xong!


In [33]:
# %% Cell 5 — Class GreedyAgent
class GreedyAgent:
    """
    Agent greedy đơn giản:
      - perceive() : quan sát trạng thái hiện tại
      - act()      : luôn chọn nước đi có Manhattan nhỏ nhất
      - run()      : nếu nước vừa đi quay ngược về bước trước → dừng ngay
    """
    def __init__(self):
        self.state = None

    def perceive(self, state):
        self.state = state

    def act(self):

        moves = get_moves(self.state)
        moves.sort(key=lambda x: manhattan(x[0]))
        new_state, direction = moves[0]
        return new_state, direction, manhattan(new_state)

    def run(self, initial_state):
        ARROW = {'UP': '↑', 'DOWN': '↓', 'LEFT': '←', 'RIGHT': '→'}

        print("=" * 45)
        print("       GREEDY AGENT GIẢI 8-PUZZLE")
        print("=" * 45)
        print_board(initial_state, "Trạng thái ban đầu:")
        print_board(GOAL_STATE,    "Trạng thái mục tiêu:")
        print(f"\n  Manhattan ban đầu: {manhattan(initial_state)}")
        print("─" * 45)

        self.perceive(initial_state)
        prev_state = None  # trạng thái ngay trước bước hiện tại
        step = 0

        while self.state != GOAL_STATE and step < MAX_STEPS:
            new_state, direction, h = self.act()
            step += 1

            print_board(new_state,
                        f"Bước {step}: {ARROW[direction]} ({direction})  │ h = {h}")

            # Ô trống vừa đi ngược lại (vd: UP rồi DOWN) → quay về prev_state → dừng
            if new_state == prev_state:
                print("\n" + "─" * 45)
                print(f"  Lặp lại: ô trống vừa quay về trạng thái bước trước — dừng!")
                print(f"   Số bước đã đi: {step}")
                print("─" * 45)
                return

            prev_state = self.state   # cập nhật prev TRƯỚC khi perceive
            self.perceive(new_state)

        print("\n" + "─" * 45)
        if self.state == GOAL_STATE:
            print(f" Agent giải thành công sau {step} bước!")
        else:
            print(f" Agent dừng sau {MAX_STEPS} bước (không tìm được lời giải).")
        print("─" * 45)

print("✅ Cell 5 xong!")

✅ Cell 5 xong!


In [ ]:
# %% Cell 6 — Chạy Agent (random tự động)
# Chỉnh SHUFFLE_STEPS để thay độ khó (càng cao càng phức tạp)
SHUFFLE_STEPS = 20

random.seed(None)  # None = hoàn toàn ngẫu nhiên mỗi lần chạy
initial = random_state(steps=SHUFFLE_STEPS)
print(f" Trạng thái random: {initial}")

agent = GreedyAgent()
agent.run(initial)